# Tone Checker: DistilBERT Toxicity Model Training (Google Colab)
This notebook fine-tunes `distilbert-base-uncased` on the Jigsaw Toxic Comment Classification dataset using PyTorch and Hugging Face Transformers.

### Features:
- GPU Acceleration (T4 / A100)
- Class-weighted Binary Cross-Entropy Loss (`BCEWithLogitsLoss`)
- 6 target categories: `toxic`, `severe_toxic`, `obscene`, `threat`, `insult`, `identity_hate`
- ROC-AUC and F1 evaluation per label
- Exports model weights for local use in Tone Checker (`/ml/artifacts/toxicity_model`)

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Install dependencies
!pip install -q transformers datasets accelerate scikit-learn pandas tqdm

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)

In [ ]:
# 3. Load Jigsaw Toxic Dataset (or sample subset)
from datasets import load_dataset

LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

try:
    print("Loading Jigsaw dataset from Hugging Face...")
    ds = load_dataset("jigsaw_toxicity_pred", split="train[:50000]")
    df = pd.DataFrame(ds)
except Exception as e:
    print(f"Fallback to direct download or synthetic set: {e}")
    # Alternative direct Kaggle link or synthetic sample
    df = pd.read_csv("https://raw.githubusercontent.com/unitaryai/detoxify/master/tests/test_comments.csv")

print("Dataset shape:", df.shape)
df.head()

In [ ]:
# 4. Dataset & DataLoaders
class ToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
texts = df["comment_text"].fillna("").tolist()
labels = df[LABELS].values

train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.15, random_state=42)
train_loader = DataLoader(ToxicDataset(train_texts, train_labels, tokenizer), batch_size=32, shuffle=True)
val_loader = DataLoader(ToxicDataset(val_texts, val_labels, tokenizer), batch_size=64, shuffle=False)

In [ ]:
# 5. Model Initialization & Class-Weighted Loss
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(LABELS),
    problem_type="multi_label_classification"
).to(device)

# Compute class weights
pos_counts = np.maximum(np.sum(train_labels, axis=0), 1)
pos_weights = torch.tensor(np.clip((len(train_labels) - pos_counts) / pos_counts, 1.0, 50.0), dtype=torch.float).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
epochs = 2
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

In [ ]:
# 6. Training & Evaluation Loop
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, targets)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    # Evaluation
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in val_loader:
            logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device)).logits
            preds.append(torch.sigmoid(logits).cpu().numpy())
            trues.append(batch["labels"].numpy())
    preds = np.vstack(preds)
    trues = np.vstack(trues)
    print(f"Epoch {epoch+1} Complete. Mean ROC-AUC: {np.mean([roc_auc_score(trues[:, i], preds[:, i]) for i in range(len(LABELS))]):.4f}")

In [ ]:
# 7. Save Model & Download Archive
OUTPUT_DIR = "./toxicity_model"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
!zip -r toxicity_model.zip ./toxicity_model

from google.colab import files
files.download("toxicity_model.zip")
print("Download initiated! Extract the zip into your local repo at /ml/artifacts/toxicity_model")